## Implementation of a simple LLM (without any enhancement on query / retrieved docs / routing)

### Libraries, ChatOllama, Chroma vectorstore, LLM prompt initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Basic LLM Testing"

# Initialize LLM
REASONING = True

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=True if REASONING else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries
client = Client(Settings())
client = chromadb.PersistentClient(path="../chroma_db")

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.list_collections()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_4722/3439006483.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


[Collection(name=vaa_documents)]

In [ ]:
# The LLM prompt
LLM_prompt = \
    """
    You are a professional academic advisor at The Hong Kong Polytechnic University. Please adhere to the following rules:
        1. Answer in the same language as the user query, e.g., English query, English answer.
        2. Avoid saying "may", "maybe", or anything similar; be affirmative, confident, and decisive in your answers.
        3. Avoid saying "based on the provided context", or anything similar; answer directly.
        4. Say no if you cannot answer the question; do not fabricate a factually false answer.
        5. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to be nice and ask if there are any more enquiries.

    *Context*:
    ----------
    {context}
    ----------

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

### Basic Implementation

In [ ]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    retrieved_docs = vectorStore.similarity_search(state["question"], k=3)
    return {"context": retrieved_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    #print(response.additional_kwargs)

    # Include the reasoning part in the output
    return {"answer": f"<think>\n{response.additional_kwargs.get("reasoning_content", "")}</think>\n\n{response.content}"}

    # NOT Include the reasoning part
    #return {"answer": f"{response.content}"}

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    graph = graph_builder.compile()

graph_building()

### Testing

In [ ]:
query = \
"What is the potential career path for studing in BEng Scheme in EE?"

print(f"Generating {query}")
result = graph.invoke({"question": query})

print(f"\nAnswer generated:\n{result['answer']}")

Generating What is the potential career path for studing in BEng Scheme in EE?

Answer generated:
<think>
Hmm, the user is asking about potential career paths for studying in the BEng Scheme in Electrical Engineering. Let me think about this carefully.

First, I need to consider what the user might really need. They're probably a student or prospective student interested in this program, and they want to understand how their studies will translate to future career opportunities. I should provide clear, factual information about career prospects.

Looking at the context provided, I see information about academic requirements and program structures, but not specifically about career paths. However, I do know that the BEng in Electrical Engineering prepares students for various roles in the electronics and electrical industries.

I should mention typical career paths like engineering roles in power systems, electronics, telecommunications, and control systems. I can also highlight that gr